# 03b_Stage2_ML

**ST498 Capstone | BDO Forward-Looking Default Risk**

## What this notebook does

Stage A forecast 12 US macroeconomic variables out to 2030 Q4. Stage B asks what those
forecasts imply for credit card defaults, and this notebook answers that with machine
learning methods.

The target is the US credit card delinquency rate (FRED: DRCCLACBS): the share of credit
card balances 30 or more days past due. The approach is a satellite model: regress the
delinquency rate on lagged macroeconomic variables, then feed the Stage A projections
through the fitted model to obtain a delinquency path to 2030 Q4. That path is what feeds
an IFRS 9 Expected Credit Loss calculation, which requires banks to base provisions on
forward-looking macroeconomic information rather than historical averages alone.

Input is `Stage1_final_regressors_US_Q.csv` from `02c_Stage1_WinnerSelection`.

## How this fits with the other Stage B notebook

`03a_Stage2_TimeSeries` covers the time-series track (ARMA, ARIMAX, SARIMAX). Both
notebooks forecast the same target over the same window from the same 2020 Q4 origin, so
the two model families can be compared directly in `03c_Stage2_WinnerSelection`.

## Method in one paragraph

Regressors are chosen by a four-method consensus vote run on training data only. Nine
models are then fitted and compared over a single evaluation origin at 2020 Q4. OLS is
pre-registered as the primary model and the deliverable, decided before any results were
seen; machine learning models are challengers. If a challenger wins, that is reported,
but OLS remains the deliverable because IFRS 9 model governance requires coefficients
that can be explained to an auditor.

Full procedure and the reasoning behind each choice: `Stage_B_ML_Methodology.pdf`.
Interpretation of results belongs in report Sections 5.2 and 6.2, not in this notebook.

| Section | Content |
|---|---|
| 0 | Imports and configuration |
| 1 | Data loading and validation |
| 2 | Joint regressor selection (four-method consensus vote) |
| 3 | Feature matrix construction (primary + robustness runs R1–R3) |
| 4 | Model fitting (nine models, inner tune/validation split) |
| 5 | Evaluation (per-horizon metrics, skill scores, Diebold-Mariano test) |
| 6 | Winner selection, production forecast, export |


## Section 0 - Imports and Configuration

All imports and constants live here, so no later cell introduces a dependency or a
magic number mid-notebook.

**Time splits.** Models are fitted on 1991 Q1 to 2020 Q4 and evaluated on 2021 Q1 to
2025 Q4, which the models never see during fitting or tuning. `TUNE_END` and `VAL_START`
divide the training block further: hyperparameters are searched on the earlier part and
chosen on the later part, keeping the evaluation window clean. `CLEAN_START` marks where
the target stops being spline-imputed and becomes genuinely observed.

**CANDIDATES.** The 12 regressors the selection vote draws from, each at the lag where
its cross-correlation with the delinquency rate peaked in `01_EDA` Section 5. The `_L2`
suffix means the variable enters two quarters before the delinquency rate it helps
explain.

**UNVALIDATED_STAGE_A.** Six of the 12 did not beat naive persistence in their own
Stage A forecasts. Any that survive the vote carry more forecast uncertainty than the
rest, and this is reported rather than treated as disqualifying.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LassoCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.base import clone

import xgboost as xgb
from scipy import stats

SEED = 42

NAVY, TEAL, AMBER = '#1F3864', '#17A589', '#E67E22'
RED, GREEN, GREY  = '#C0392B', '#27AE60', '#7F8C8D'
BLUE, LBLUE       = '#2E75B6', '#AED6F1'
TEMPLATE = 'plotly_white'

GITHUB_RAW     = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'
REGRESSOR_FILE = f'{GITHUB_RAW}/Stage1_final_regressors_US_Q.csv'
OUT_DIR        = Path.cwd().parent / 'Stage1_Outputs'

TARGET     = 'us_delinquency_rate'
TARGET_RAW = 'us_delinquency_rate_raw'

TRAIN_START, TRAIN_END  = '1991-03-31', '2020-12-31'
TUNE_END                = '2016-12-31'
VAL_START,   VAL_END    = '2017-03-31', '2020-12-31'
EVAL_START,  EVAL_END   = '2021-03-31', '2025-12-31'
CLEAN_START, CLEAN_END  = '2022-09-30', '2025-12-31'
FCST_START,  FCST_END   = '2026-03-31', '2030-12-31'
COVID_START, COVID_END  = '2020-03-31', '2022-06-30'

HORIZONS = [1, 4, 8, 12, 20]

# Each variable at its peak CCF lag from 01_EDA Section 5.
# Bond yield enters as first difference: the level failed the EDA stationarity tests.
CANDIDATES = [
    'us_gdp_yoy_growth_L2',
    'us_unemployment_L0',
    'us_cpi_L0',
    'us_consumer_confidence_L2',
    'us_bond_yield_10y_d1_L2',
    'us_credit_qoq_growth_L6',
    'us_sp500_log_ret_L4',
    'us_vix_log_ret_L0',
    'us_house_price_yoy_L3',
    'us_indprod_yoy_L3',
    'us_oil_yoy_L2',
    'us_reer_diff_L0',
]

# CPI L6 competes only in the lag-rich robustness run (R2), not the primary vote.
CANDIDATES_R2 = CANDIDATES + ['us_cpi_L6']

# Stage A winners that failed the DM test against naive persistence (02c).
UNVALIDATED_STAGE_A = [
    'us_consumer_confidence', 'us_credit_qoq_growth', 'us_indprod_yoy',
    'us_reer_diff', 'us_sp500_log_ret', 'us_vix_log_ret',
]

print(f'Configuration loaded | {len(CANDIDATES)} primary candidates | seed {SEED}')
print(f'Output directory: {OUT_DIR}')

## Section 1 - Data Loading and Validation

Input is `Stage1_final_regressors_US_Q.csv` from `02c`: 164 quarterly rows, 1990 Q1 to
2030 Q4. Rows to 2025 Q4 are historical observations; 2026 Q1 onward are Stage A
projections. The load raises rather than warns if a column is missing or the row count is
unexpected, so a stale input file cannot reach the results.

**Two target columns.** During 2020 Q1 to 2022 Q2, stimulus payments and loan moratoria
held observed delinquency well below where macroeconomic conditions alone would have put
it; the EBA (2021, paragraph 75) documents that public support measures reduced observed
defaults. A model trained on those values learns a weaker macro-to-default relationship
than really holds, which is the opposite of what IFRS 9 requires. `us_delinquency_rate` is
therefore reconstructed over that window by a cubic spline fitted through the surrounding
non-COVID observations in `01_EDA`, and is the modelling target.
`us_delinquency_rate_raw` keeps the original series and is used in robustness run R3, which
tests whether the adjustment matters. The adjustment is contested — Liu et al. (2025)
exclude these quarters instead of reconstructing them.

**Calendar quarters vs usable quarters.** A quarter is usable only if the target and every
regressor are observed. Lagged variables lose their leading quarters, and REER has no data
before 1994 Q2, so requiring all 12 candidates costs 13 of the 120 training quarters and
the usable sample starts at 1994 Q2. Table 1.2 shows which variables bind. The usable
sample therefore depends on the feature set, and the report quotes these figures rather
than the calendar length.

In [ ]:
df = pd.read_csv(REGRESSOR_FILE, index_col=0, parse_dates=True).sort_index()

required = CANDIDATES + [TARGET, TARGET_RAW, 'covid_dummy']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f'Input file missing required columns: {missing}')
if len(df) != 164:
    raise ValueError(f'Expected 164 rows from 02c, found {len(df)}')

print(f'Loaded {df.shape[0]} rows x {df.shape[1]} columns '
      f'({df.index.min().date()} to {df.index.max().date()})')
print(f'All {len(required)} required columns present.')
if 'delinquency_spline' in df.columns:
    print('NOTE: delinquency_spline still present - 02c drop not applied in this version.')

df_train = df.loc[TRAIN_START:TRAIN_END].copy()
df_tune  = df.loc[TRAIN_START:TUNE_END].copy()
df_val   = df.loc[VAL_START:VAL_END].copy()
df_eval  = df.loc[EVAL_START:EVAL_END].copy()
df_clean = df.loc[CLEAN_START:CLEAN_END].copy()
df_full  = df.loc[TRAIN_START:EVAL_END].copy()
df_fcst  = df.loc[FCST_START:FCST_END].copy()


def q(ts):
    return f'{ts.year} Q{ts.quarter}'

blocks = [('Training block', df_train), ('Tune split', df_tune), ('Val split', df_val),
          ('Evaluation', df_eval), ('Clean sub-window', df_clean), ('Full panel', df_full)]

print('\nTime splits (calendar quarters | usable with all 12 candidates):')
for name, b in blocks:
    n_cal = len(b)
    n_use = len(b[CANDIDATES + [TARGET]].dropna())
    print(f'  {name:<17}: {q(b.index.min())} to {q(b.index.max())}   {n_cal:>3} | {n_use:>3}')

print(f'  {"Forecast":<17}: {q(df_fcst.index.min())} to {q(df_fcst.index.max())}   '
      f'{len(df_fcst):>3} | {len(df_fcst[CANDIDATES].dropna()):>3}')

In [ ]:
starts = pd.DataFrame([
    {'Variable': c,
     'First valid': df[c].first_valid_index().date(),
     'NaNs in training block': int(df_train[c].isna().sum())}
    for c in CANDIDATES + [TARGET]
]).sort_values('First valid', ascending=False)

print('Table 1.2 - First valid observation per candidate')
display(starts.set_index('Variable'))

binding = df_train[CANDIDATES + [TARGET]].isna().any(axis=1)
print(f'Training rows lost to listwise deletion: {binding.sum()} of {len(df_train)}')
print(f'First complete case: {df_train[~binding].index.min().date()}')

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET_RAW],
    mode='lines', name='Raw observed',
    line=dict(color=NAVY, width=2)))

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET],
    mode='lines', name='Spline-adjusted (model target)',
    line=dict(color=AMBER, width=2, dash='dot')))

# COVID spline window: target is reconstructed, not observed
fig.add_vrect(
    x0=COVID_START, x1=COVID_END,
    fillcolor='rgba(192,57,43,0.10)', line_width=0,
    annotation_text='COVID spline window', annotation_position='top left',
    annotation_font=dict(size=9, color=RED))

# Clean sub-window: evaluation quarters scored against genuinely observed values
fig.add_vrect(
    x0=CLEAN_START, x1=CLEAN_END,
    fillcolor='rgba(46,117,182,0.08)', line_width=0,
    annotation_text='Clean sub-window', annotation_position='top right',
    annotation_font=dict(size=9, color=BLUE))

# Evaluation origin. Annotation added separately: add_vline's own annotation breaks on
# date-string axes in older plotly (it averages the x coords to place the label).
fig.add_vline(x=EVAL_START, line_dash='dash', line_color=GREY, line_width=1.2)
fig.add_annotation(
    x=EVAL_START, y=0.02, yref='paper', text='Evaluation origin',
    showarrow=False, font=dict(size=9, color=GREY),
    xanchor='left', xshift=4, bgcolor='rgba(255,255,255,0.7)')

train_mean = df_train[TARGET].mean()
fig.add_hline(y=train_mean, line_dash='dot', line_color=GREY, line_width=1,
              annotation_text=f'Training mean = {train_mean:.2f}%',
              annotation_position='bottom right',
              annotation_font=dict(size=9, color=GREY))

fig.update_layout(
    title=dict(
        text=('<b>Figure 1.1 - Target Variable: Raw vs Spline-Adjusted</b>'
              f'<br><span style="font-size:11.5px;color:{GREY}">'
              'Red = spline reconstruction window | Blue = clean evaluation sub-window | '
              'the 6 quarters between them are scored against imputed values</span>'),
        font=dict(size=16, color=NAVY), x=0.015, xanchor='left', y=0.96, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='top', y=1.15, xanchor='center', x=0.5),
    margin=dict(t=115, b=55, l=70, r=30))
fig.show()

## Section 2 - Joint Regressor Selection

The report's earlier approach kept a variable if its univariate cross-correlation with the
delinquency rate cleared a significance threshold. A variable can pass that screen and
still be redundant, or change sign, once correlated variables enter alongside it (Bellotti
and Crook 2013, Tables 3 and 4). Selection is therefore run inside a multivariate
framework instead.

No single method is reliable on its own: stepwise depends on removal order, Lasso picks
arbitrarily among correlated variables, and tree importances favour variables with more
split points. Four methods vote instead, and a variable is kept if at least two nominate
it.

**The four methods (2.2 to 2.5).** Backward stepwise by AIC removes the least informative
variable until AIC stops improving. Lasso ranks variables by coefficient magnitude at a
cross-validated penalty. Random Forest permutation importance measures the accuracy drop
when a variable is shuffled. Gradient boosting ranks by contribution to reducing training
error. Each nominates five of the twelve candidates.

**Rules fixed in advance.** Selection runs on the training block only, so the evaluation
window cannot influence which variables are chosen. `covid_dummy` is forced into every
model as a structural break control and never competes. CPI competes on the same terms as
everything else. If the vote returned fewer than four or more than eight variables, a
pre-specified tie-break takes the top six by vote count then mean rank.

The retained set defines Equation 5.11 in the report. It is a result, reported in Section
6, not a methodology pre-specification.

**Sensitivity (2.7, 2.8).** Two checks test whether the outcome depends on arbitrary
choices. The first re-runs the vote without REER, which has no data before 1994 Q2 and
truncates the selection sample by 13 quarters. The second varies how many variables each
method nominates. Both are reported as sensitivity; neither revises the primary set.

**Diagnostics (2.9 to 2.12).** These feed no selection decision. All coefficient
comparisons use one fixed sample, `COMMON_IDX`, so that a coefficient moving between
specifications reflects the change in specification and not a change in which quarters are
included — without this, adding a variable with a shorter history alters the sample and the
conditioning at the same time. Table 2.3 is the deliberate exception, reported on the
selected model's own sample, because variance inflation should describe the model actually
fitted. Cell 2.12 traces one coefficient across nested specifications to locate where its
sign changes.

In [ ]:
# 2.1 - selection sample and vote parameters
train_sel = df_train[CANDIDATES + [TARGET, 'covid_dummy']].dropna()

X_sel = train_sel[CANDIDATES]
y_sel = train_sel[TARGET].values

# Scaled copy shared by the Lasso and tree methods. Built once here so the four
# selection cells below can run in any order.
scaler_sel = StandardScaler()
X_sel_s    = scaler_sel.fit_transform(X_sel)

TOP_N          = 5   # each method nominates this many
VOTE_THRESHOLD = 2   # retained if nominated by at least this many methods

print(f'Selection sample: {len(train_sel)} quarters '
      f'({q(train_sel.index.min())} to {q(train_sel.index.max())})')
print(f'{len(CANDIDATES)} candidates | each method nominates {TOP_N} | '
      f'retained at >= {VOTE_THRESHOLD} of 4 votes')

In [ ]:
# 2.2 - method 1 backward stepwise AIC
def backward_stepwise_aic(X_df, y, feature_cols, top_n):
    """Backward elimination by AIC. Starts with all candidates and drops the variable
    whose removal most improves AIC, stopping at top_n or when no removal helps."""
    remaining = list(feature_cols)
    while len(remaining) > top_n:
        full_aic = sm.OLS(y, sm.add_constant(X_df[remaining].values)).fit().aic
        best_aic, to_remove = full_aic, None
        for feat in remaining:
            others = [f for f in remaining if f != feat]
            aic = sm.OLS(y, sm.add_constant(X_df[others].values)).fit().aic
            if aic < best_aic:
                best_aic, to_remove = aic, feat
        if to_remove is None:
            break
        remaining.remove(to_remove)
    return remaining[:top_n]

top5_aic = backward_stepwise_aic(X_sel, y_sel, CANDIDATES, TOP_N)

print(f'Backward AIC top {len(top5_aic)}:')
for v in top5_aic:
    print(f'  {v}')

In [ ]:
# 2.3 - method 2 Lasso
lasso_cv = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000, random_state=SEED)
lasso_cv.fit(X_sel_s, y_sel)

lasso_ranked = sorted(
    [(c, abs(w)) for c, w in zip(CANDIDATES, lasso_cv.coef_) if abs(w) > 1e-6],
    key=lambda t: t[1], reverse=True)
top5_lasso = [c for c, _ in lasso_ranked[:TOP_N]]

print(f'Lasso (alpha={lasso_cv.alpha_:.4f}) nominated {len(lasso_ranked)} '
      f'non-zero, top {len(top5_lasso)} by |coefficient|:')
for c, w in lasso_ranked[:TOP_N]:
    print(f'  {c}: {w:.4f}')

In [ ]:
# 2.4 - method 3 Random Forest permutation importance
rf_sel = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1)
rf_sel.fit(X_sel_s, y_sel)

perm    = permutation_importance(rf_sel, X_sel_s, y_sel, n_repeats=20,
                                 random_state=SEED, scoring='neg_mean_absolute_error')
rf_imp  = dict(zip(CANDIDATES, perm.importances_mean))
top5_rf = sorted(rf_imp, key=rf_imp.get, reverse=True)[:TOP_N]

print(f'Random Forest permutation importance top {TOP_N}:')
for v in top5_rf:
    print(f'  {v}: {rf_imp[v]:.4f}')

In [ ]:
# 2.5 - method 4 Gradient Boosting importance
gb_sel = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                   learning_rate=0.05, random_state=SEED)
gb_sel.fit(X_sel_s, y_sel)

gb_imp  = dict(zip(CANDIDATES, gb_sel.feature_importances_))
top5_gb = sorted(gb_imp, key=gb_imp.get, reverse=True)[:TOP_N]

print(f'Gradient Boosting importance top {TOP_N}:')
for v in top5_gb:
    print(f'  {v}: {gb_imp[v]:.4f}')

In [ ]:
# 2.6 - Table 2.1 consensus vote
nominations = {'AIC': top5_aic, 'Lasso': top5_lasso, 'RF': top5_rf, 'GB': top5_gb}

# Rank within each method's top-N; variables not nominated get TOP_N + 1.
vote_counts, mean_ranks = {}, {}
for c in CANDIDATES:
    ranks = [(m.index(c) + 1) if c in m else TOP_N + 1 for m in nominations.values()]
    vote_counts[c] = sum(1 for m in nominations.values() if c in m)
    mean_ranks[c]  = np.mean(ranks)

vote_df = pd.DataFrame({
    'Variable':  CANDIDATES,
    'Votes':     [vote_counts[c] for c in CANDIDATES],
    'Mean rank': [round(mean_ranks[c], 2) for c in CANDIDATES],
    **{m: ['Yes' if c in lst else 'No' for c in CANDIDATES]
       for m, lst in nominations.items()},
}).sort_values(['Votes', 'Mean rank'], ascending=[False, True]).reset_index(drop=True)

print('Table 2.1 - Four-method consensus vote')
display(vote_df.set_index('Variable'))

consensus = vote_df.loc[vote_df['Votes'] >= VOTE_THRESHOLD, 'Variable'].tolist()

# Pre-specified tie-break: if the vote returns fewer than 4 or more than 8, fall back to
# the top 6 by vote count then mean rank. Recorded whether or not it fires.
if len(consensus) < 4 or len(consensus) > 8:
    consensus = vote_df['Variable'].head(6).tolist()
    print('\nTie-break applied: vote returned an out-of-range set, top 6 taken.')
else:
    print(f'\nTie-break not triggered ({len(consensus)} variables, within 4-8).')

FEATURES_EQ511         = consensus + ['covid_dummy']
FEATURES_EQ511_NODUMMY = consensus

unval = [c for c in consensus if any(c.startswith(u) for u in UNVALIDATED_STAGE_A)]

print(f'\nRetained ({len(consensus)} of {len(CANDIDATES)}):')
for c in consensus:
    print(f'  {c}{"  [Stage A unvalidated]" if c in unval else ""}')
print(f'\nRejected: {[c for c in CANDIDATES if c not in consensus]}')
print(f'\nThis set defines Equation 5.11. The equation is a result, reported in '
      f'Section 6, not a methodology pre-specification.')
print(f'{len(unval)} of {len(consensus)} retained variables lack Stage A DM validation.')

In [ ]:
# 2.7 - sensitivity REER excluded
def run_vote(candidates, df_block, top_n=TOP_N, threshold=VOTE_THRESHOLD):
    """Full four-method vote on an arbitrary candidate list. Used for sensitivity checks;
    the primary vote above is run cell by cell so each method's output is visible."""
    s  = df_block[candidates + [TARGET]].dropna()
    Xd = s[candidates]
    y  = s[TARGET].values
    Xs = StandardScaler().fit_transform(Xd)

    a = backward_stepwise_aic(Xd, y, candidates, top_n)

    lc = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000,
                 random_state=SEED).fit(Xs, y)
    l  = [c for c, _ in sorted([(c, abs(w)) for c, w in zip(candidates, lc.coef_)
                                if abs(w) > 1e-6],
                               key=lambda t: t[1], reverse=True)[:top_n]]

    rf = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1).fit(Xs, y)
    pi = permutation_importance(rf, Xs, y, n_repeats=20, random_state=SEED,
                                scoring='neg_mean_absolute_error')
    r  = [candidates[i] for i in np.argsort(pi.importances_mean)[::-1][:top_n]]

    gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                                   random_state=SEED).fit(Xs, y)
    g  = [candidates[i] for i in np.argsort(gb.feature_importances_)[::-1][:top_n]]

    counts = {c: sum(c in lst for lst in [a, l, r, g]) for c in candidates}
    return sorted([c for c in candidates if counts[c] >= threshold],
                  key=lambda c: -counts[c]), len(s)

# Guard: run_vote reimplements the four methods, so confirm it reproduces the explicit
# vote above before any sensitivity result is trusted.
_check, _ = run_vote(CANDIDATES, df_train)
assert set(_check) == set(consensus), 'run_vote diverges from the explicit primary vote'

# REER truncates the selection sample by 13 quarters (no data before 1994 Q2). The
# justification for re-running without it is data availability, which is knowable without
# reference to any vote outcome, so this is not result-dependent selection.
cand_no_reer     = [c for c in CANDIDATES if not c.startswith('us_reer')]
sens_set, sens_n = run_vote(cand_no_reer, df_train)

print(f'Primary vote : {len(train_sel)} quarters, {len(consensus)} retained')
print(f'Without REER : {sens_n} quarters, {len(sens_set)} retained')
print(f'\nSensitivity set: {sens_set}')

added   = [c for c in sens_set if c not in consensus]
dropped = [c for c in consensus if c not in sens_set]
if not added and not dropped:
    print('\nIdentical to the primary vote. REER truncation does not affect selection.')
else:
    print(f'\nDiffers from primary vote. Added: {added} | Dropped: {dropped}')
    print('This difference must be reported in Section 6.')

In [ ]:
# 2.8 - sensitivity nomination width
# TOP_N = 5 is the pre-registered value. This records how the retained set would change
# under alternatives. Reported as sensitivity; it does not revise the primary set.
print(f'{"top_n":>6} {"Kept":>5}  Variables')
for tn in [4, 5, 6, 7]:
    s, _ = run_vote(CANDIDATES, df_train, top_n=tn)
    mark = '  <- pre-registered' if tn == TOP_N else ''
    print(f'{tn:>6} {len(s):>5}  {[c.replace("us_", "") for c in s]}{mark}')

In [ ]:
# 2.9 - common diagnostic sample
# All diagnostic fits are restricted to a single sample so that coefficient comparisons
# reflect specification changes only, not sample composition. Using the all-12 complete
# case index means every nested fit below sees identical quarters.
COMMON_IDX = df_train[CANDIDATES + [TARGET, 'covid_dummy']].dropna().index
print(f'Common diagnostic sample: {len(COMMON_IDX)} quarters '
      f'({q(COMMON_IDX.min())} to {q(COMMON_IDX.max())})')

In [ ]:
# 2.10 - Table 2.2 joint sign diagnostic
# Economic priors set before fitting. Ambiguous cases are recorded as such rather than
# assigned a direction, so the check cannot be passed by hindsight.
SIGN_PRIORS = {
    'us_gdp_yoy_growth_L2':      ('-', 'growth reduces defaults'),
    'us_unemployment_L0':        ('+', 'joblessness raises defaults'),
    'us_cpi_L0':                 ('?', 'erodes real debt burden but also real income'),
    'us_consumer_confidence_L2': ('-', 'confidence reduces defaults'),
    'us_bond_yield_10y_d1_L2':   ('+', 'rising rates raise debt service'),
    'us_credit_qoq_growth_L6':   ('+', 'credit expansion precedes over-leverage'),
    'us_sp500_log_ret_L4':       ('-', 'equity gains support household balance sheets'),
    'us_vix_log_ret_L0':         ('+', 'volatility signals stress'),
    'us_house_price_yoy_L3':     ('-', 'housing wealth and collateral support repayment'),
    'us_indprod_yoy_L3':         ('-', 'activity reduces defaults'),
    'us_oil_yoy_L2':             ('+', 'energy costs squeeze disposable income'),
    'us_reer_diff_L0':           ('?', 'competitiveness channel is indirect'),
}

def fit_hac(features, index=COMMON_IDX):
    s = df_train.loc[index, features + [TARGET]]
    X = sm.add_constant(s[features].values, has_constant='add')
    return sm.OLS(s[TARGET].values, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

ols_all12 = fit_hac(CANDIDATES + ['covid_dummy'])
ols_sel   = fit_hac(FEATURES_EQ511)
sel_coefs = dict(zip(FEATURES_EQ511, ols_sel.params[1:]))

rows = []
for i, c in enumerate(CANDIDATES):
    b12   = ols_all12.params[i + 1]
    prior = SIGN_PRIORS[c][0]
    got   = '+' if b12 > 0 else '-'
    b7    = sel_coefs.get(c, np.nan)
    rows.append({
        'Variable':      c,
        'Prior':         prior,
        'All 12':        round(b12, 4),
        'p':             round(ols_all12.pvalues[i + 1], 4),
        'Selected 7':    round(b7, 4) if c in sel_coefs else '-',
        'Matches prior': 'n/a' if prior == '?' else ('Yes' if got == prior else 'NO'),
        'Sign stable':   ('-' if c not in sel_coefs
                          else 'Yes' if np.sign(b12) == np.sign(b7) else 'FLIP'),
    })

print(f'Table 2.2 - Joint sign diagnostic (Newey-West HAC, maxlags=4, n={len(COMMON_IDX)}). '
      f'Diagnostic only.')
display(pd.DataFrame(rows).set_index('Variable'))

print(f'Joint 12-variable fit: R2={ols_all12.rsquared:.3f} | '
      f'DW={durbin_watson(ols_all12.resid):.3f} | AIC={ols_all12.aic:.1f}')


viol = [r['Variable'] for r in rows if r['Matches prior'] == 'NO']
flip = [r['Variable'] for r in rows if r['Sign stable'] == 'FLIP']
print(f'Prior violations: {viol if viol else "none"}')
print(f'Sign flips between the 12-variable and 7-variable fits: {flip if flip else "none"}')

In [ ]:
# 2.11 - Table 2.3 multicollinearity
# Reported on the selected model's own sample, not COMMON_IDX: variance inflation should
# describe the model actually fitted, so n here exceeds the diagnostic sample above.
vif_data = df_train[FEATURES_EQ511].dropna()
vif_mat  = sm.add_constant(vif_data.values, has_constant='add')

vif_df = pd.DataFrame({
    'Variable': list(vif_data.columns),
    'VIF': [round(variance_inflation_factor(vif_mat, i + 1), 2)
            for i in range(vif_data.shape[1])],
}).sort_values('VIF', ascending=False).reset_index(drop=True)

# Thresholds from Bellotti and Crook (2012), Section 3.2: below 5 for OLS, below 10 for
# regularised and nonlinear models, which tolerate correlated inputs.
vif_df['Assessment'] = ['OK for OLS' if v < 5 else
                        'OK for regularised only' if v < 10 else 'High'
                        for v in vif_df['VIF']]

print(f'Table 2.3 - Variance Inflation Factors (n={len(vif_data)})')
display(vif_df.set_index('Variable'))

high = vif_df.loc[vif_df['VIF'] >= 5, 'Variable'].tolist()
print(f'Above 5 (OLS threshold): {high if high else "none"}')

In [ ]:
# 2.12 - GDP sign trace
# GDP enters positive against a negative prior, significantly. This traces where the
# reversal happens: alone, then conditioned on the other activity measures. All four fits
# use COMMON_IDX so the coefficient movement is attributable to conditioning alone.
gdp, unemp, indprod = 'us_gdp_yoy_growth_L2', 'us_unemployment_L0', 'us_indprod_yoy_L3'

for label, feats in [('GDP alone', [gdp]),
                     ('GDP + unemployment', [gdp, unemp]),
                     ('GDP + unemp + indprod', [gdp, unemp, indprod]),
                     ('Full selected set', FEATURES_EQ511)]:
    sub = df_train.loc[COMMON_IDX, feats + [TARGET]]
    m = sm.OLS(sub[TARGET].values,
               sm.add_constant(sub[feats].values, has_constant='add')
               ).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    k = feats.index(gdp) + 1          # +1 for the intercept
    print(f'{label:<24} GDP coef = {m.params[k]:+.4f}  '
          f'(p={m.pvalues[k]:.4f}, n={len(sub)})')

s       = df_train.loc[COMMON_IDX, [gdp, TARGET]]
vif_gdp = vif_df.loc[vif_df['Variable'] == gdp, 'VIF'].iloc[0]
print(f'\nPearson correlation, GDP vs delinquency: {s[gdp].corr(s[TARGET]):+.4f}')
print(f'GDP variance explained by other selected regressors: {1 - 1/vif_gdp:.1%}')

## Section 3 - Feature Matrix Construction

Four specifications are fitted. Each changes exactly one thing relative to the primary run,
so that a divergence in forecasts can be attributed to that one choice. IFRS 9 asks
institutions to show their ECL estimates are not critically dependent on arbitrary
modelling decisions, and the three choices tested here are either contested in the
literature or unsupported by a direct citation.

| Run | Target | Features | Dummy | What it tests |
|---|---|---|---|---|
| Primary | Spline-adjusted | 7 voted | On | Main result |
| R1 | Spline-adjusted | 7 voted | Off | Is the COVID dummy doing the work the macro variables should? |
| R2 | Spline-adjusted | All 13 candidates | On | Does excluding five variables cost accuracy? Does CPI's lag matter? |
| R3 | Raw, COVID excluded | 7 voted | n/a | Does the spline reconstruction matter, or would deletion do? |

**Complete cases.** Every matrix drops quarters where the target or any feature is missing,
so sample size varies by specification. The builder asserts that the evaluation and
forecast blocks are complete, since a dropped row in either would silently shorten the
forecast horizon or misalign forecasts against benchmarks in Section 5.

**R2 is an unrestricted specification, not a lag-rich one.** The Stage A output holds one
lag per variable plus a second CPI lag, so multiple lags per variable are not available to
test. R2 therefore uses all 13 candidate columns with no vote applied. The Lasso reduction
step is retained but its penalty does not bind on this data, so no reduction occurs.

**R2 also changes the sample.** Reintroducing REER, which has no data before 1994 Q2, costs
10 training quarters. Cell 3.6 builds the primary specification on R2's sample so the
comparison in Section 5 isolates specification from sample composition.

**R3 drops `covid_dummy`.** With the COVID quarters removed the dummy would be constant zero
and the design matrix singular. Note also that the training block ends 2020 Q4 while the
COVID window runs to 2022 Q2, so only 4 training quarters are affected; the remaining 6
fall inside the evaluation window. R3 is therefore a narrower test than the ten-quarter
window suggests. Because the spline only alters 2020 Q1 to 2022 Q2, the adjusted and raw
targets coincide from 2022 Q3, and R3 and the primary run are scored against identical
values on the clean sub-window.

**The dummy is identified off 4 observations.** This makes R1 a substantive test rather than
a formality, and the imprecision is a limitation in its own right regardless of what R1
shows.

In [ ]:
# 3.1 - matrix builder
def build_matrices(features, target, blocks=None):
    """Aligned X, y arrays for every block a run needs.

    Complete cases only: a quarter is dropped if the target or any feature is missing.
    Returns a dict keyed by block name holding (X, y, index), except 'fcst' which holds
    (X, index) since no target exists beyond 2025 Q4. Pass `blocks` to override any
    default block, which is how R3 substitutes its COVID-excluded training data.
    """
    B = {'train': df_train, 'tune': df_tune, 'val': df_val,
         'eval': df_eval, 'full': df_full, 'fcst': df_fcst}
    if blocks:
        B.update(blocks)

    def prep(block):
        s = block[features + [target]].dropna()
        return s[features].values, s[target].values, s.index

    m = {k: prep(B[k]) for k in ['train', 'tune', 'val', 'eval', 'full']}
    s_fc = B['fcst'][features].dropna()
    m['fcst'] = (s_fc.values, s_fc.index)

    # Evaluation and forecast blocks must be complete. A dropped row in either would
    # silently shorten the horizon or misalign forecasts against benchmarks in Section 5.
    assert len(m['eval'][2]) == len(B['eval']), \
        f'eval incomplete: {len(m["eval"][2])} of {len(B["eval"])}'
    assert len(m['fcst'][1]) == len(B['fcst']), \
        f'forecast incomplete: {len(m["fcst"][1])} of {len(B["fcst"])}'
    return m

print('build_matrices defined.')

In [ ]:
# 3.2 - primary run and R1
mats_primary = build_matrices(FEATURES_EQ511, TARGET)
mats_r1      = build_matrices(FEATURES_EQ511_NODUMMY, TARGET)

for name, m, feats in [('Primary', mats_primary, FEATURES_EQ511),
                       ('R1', mats_r1, FEATURES_EQ511_NODUMMY)]:
    print(f'{name:<8} {len(feats)} features | train {len(m["train"][1])} | '
          f'full {len(m["full"][1])} | eval {len(m["eval"][1])} | fcst {len(m["fcst"][1])}')

In [ ]:
# 3.3 - R2 unrestricted specification
# All 13 available candidate columns, no vote. The Stage A output holds one lag per
# variable plus a second CPI lag, so this tests whether the vote's exclusion of five
# variables costs accuracy, and whether CPI's lag choice matters.
r2_train = df_train[CANDIDATES_R2 + [TARGET]].dropna()
X_r2     = StandardScaler().fit_transform(r2_train[CANDIDATES_R2])
y_r2     = r2_train[TARGET].values

lasso_r2 = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000,
                   random_state=SEED).fit(X_r2, y_r2)
kept_r2  = [c for c, w in zip(CANDIDATES_R2, lasso_r2.coef_) if abs(w) > 1e-6]

print(f'R2 Lasso alpha={lasso_r2.alpha_:.4f} | '
      f'retained {len(kept_r2)} of {len(CANDIDATES_R2)}')
if len(kept_r2) == len(CANDIDATES_R2):
    print('Penalty does not bind, so no reduction occurs. R2 is the full unrestricted set.')
else:
    print(f'Dropped: {[c for c in CANDIDATES_R2 if c not in kept_r2]}')

FEATURES_SPECD = kept_r2 + ['covid_dummy']
mats_r2        = build_matrices(FEATURES_SPECD, TARGET)

print(f'\nR2       {len(FEATURES_SPECD)} features | train {len(mats_r2["train"][1])} | '
      f'full {len(mats_r2["full"][1])}')

In [ ]:
# 3.4 - R3 raw target, COVID quarters excluded
def drop_covid(block):
    return block[~((block.index >= COVID_START) & (block.index <= COVID_END))]

# covid_dummy is removed rather than left in: with the COVID quarters gone it would be
# constant zero across training and the design matrix singular.
FEATURES_R3 = [f for f in FEATURES_EQ511 if f != 'covid_dummy']

mats_r3 = build_matrices(FEATURES_R3, TARGET_RAW, blocks={
    'train': drop_covid(df_train),
    'tune':  drop_covid(df_tune),
    'val':   drop_covid(df_val),
    'full':  drop_covid(df_full),
})

# The spline only modifies 2020 Q1 to 2022 Q2, so on the clean sub-window the adjusted and
# raw targets coincide. R3 and the primary run are therefore scored against identical
# values there, and any difference between them is due to training treatment alone.
clean_same = np.allclose(df.loc[CLEAN_START:CLEAN_END, TARGET],
                         df.loc[CLEAN_START:CLEAN_END, TARGET_RAW])

print(f'R3       {len(FEATURES_R3)} features | train {len(mats_r3["train"][1])} | '
      f'full {len(mats_r3["full"][1])} (COVID quarters excluded)')
print(f'Training quarters removed: {len(df_train) - len(drop_covid(df_train))}')
print(f'Spline and raw targets identical on the clean sub-window: {clean_same}')

In [ ]:
# 3.5 - Table 3.1 run register
RUNS = {
    'Primary': dict(mats=mats_primary, features=FEATURES_EQ511,        target=TARGET,
                    dummy='On',  tests='main result'),
    'R1':      dict(mats=mats_r1,      features=FEATURES_EQ511_NODUMMY, target=TARGET,
                    dummy='Off', tests='is the COVID dummy driving results'),
    'R2':      dict(mats=mats_r2,      features=FEATURES_SPECD,         target=TARGET,
                    dummy='On',  tests='does excluding five variables cost accuracy'),
    'R3':      dict(mats=mats_r3,      features=FEATURES_R3,            target=TARGET_RAW,
                    dummy='n/a', tests='does the spline adjustment matter'),
}

run_df = pd.DataFrame([
    {'Run': k,
     'Target': 'spline' if v['target'] == TARGET else 'raw',
     'Features': len(v['features']),
     'Dummy': v['dummy'],
     'Train n': len(v['mats']['train'][1]),
     'Full n': len(v['mats']['full'][1]),
     'What it tests': v['tests']}
    for k, v in RUNS.items()
])

print('Table 3.1 - Robustness run register')
display(run_df.set_index('Run'))

In [ ]:
# 3.6 - sample overlap between runs
# R2 reintroduces REER and so starts at 1994 Q2, giving it 10 fewer training quarters than
# the primary run. A supplementary matrix fits the primary specification on R2's sample so
# that the R2 comparison in Section 5 isolates specification from sample.
mats_primary_r2sample = build_matrices(
    FEATURES_EQ511, TARGET,
    blocks={'train': df_train.loc[mats_r2['train'][2]],
            'tune':  df_tune.loc[df_tune.index.intersection(mats_r2['train'][2])],
            'val':   df_val.loc[df_val.index.intersection(mats_r2['train'][2])],
            'full':  df_full.loc[mats_r2['full'][2]]})

dummy_n = int(df_train.loc[mats_primary['train'][2], 'covid_dummy'].sum())
print(f'covid_dummy = 1 on {dummy_n} of {len(mats_primary["train"][1])} training quarters')
print(f'Primary on R2 sample: train {len(mats_primary_r2sample["train"][1])} '
      f'(vs {len(mats_primary["train"][1])} on its own sample)')

## Section 4 - Model Fitting

Nine models are fitted on the primary specification: OLS, three regularised linear models,
two kernel methods, two tree ensembles, and a small neural network. OLS is the
pre-registered primary model and the deliverable; the other eight are challengers.

**Single evaluation origin.** Models are fitted once on 1991-2020 and forecast 20 quarters
forward without retraining, matching how the model would be used in production. A
rolling-origin backtest would give more test observations but answers a different question
- one-step-ahead accuracy under continual refitting - and is not comparable to a
20-quarter projection. The cost is that only one error path is observed, which limits
power and is why Section 5 applies a small-sample correction to the Diebold-Mariano test.

**Inner tune/validation split.** Eight of the nine models have hyperparameters that cannot
be estimated in one pass. The training block is divided at 2016 Q4: configurations are
fitted on 1991-2016 and scored on 2017-2020, and the winner is refit on the whole training
block. The evaluation window is never used to select a configuration.

Note what the validation window is. Delinquency was flat and historically low across 2017
to 2020, so a model that predicts close to a constant scores well on it. Validation MAE
therefore partly measures how little a model moves rather than how well it forecasts, and
it does not predict performance on the 2021-2025 evaluation window. The MLP is the clearest
case: its validation MAE is lower than its training MAE, which happens because a large L2
penalty shrinks it toward a near-flat prediction that suits a flat validation period but
misses the 2009 peak entirely.

**No k-fold cross-validation.** k-fold requires uncorrelated residuals. Figure 4.2 gives
DW=0.94 and Ljung-Box p<0.0001, so residuals are strongly serially correlated. Bergmeir,
Hyndman and Koo (2018) show k-fold is valid only for purely autoregressive models with
uncorrelated errors, so rolling and expanding splits are used throughout.

**Grid boundaries.** A hyperparameter winning at the edge of its grid means the search space
was too narrow, so the fitter warns and the grid is widened. Ridge, Lasso and Elastic Net
all select the smallest penalty available and converge on the OLS solution, with validation
MAEs identical to three decimals; extending those grids downward is not informative, so the
boundary hit is reported as a result. Parameters with a hard theoretical bound, such as
`l1_ratio` and `subsample` at 1.0, are exempt from the check.

**Two fits per model.** Each is fitted on the training block, whose forecasts are scored in
Section 5, and separately on the full history to 2025 Q4, which generates the 2026-2030
projection in Section 6. Scalers are fitted within each, on training data only.

**Figure 4.1** compares training fit against validation MAE for four representative models.
A close training fit with a high validation MAE indicates overfitting rather than skill.
XGBoost fits the training data almost exactly and generalises far worse, which is the
expected outcome for a boosted ensemble on 117 observations (Grinsztajn et al. 2022).

**Figure 4.2** shows OLS residuals. They run predominantly negative after 2011, meaning the
model over-predicts delinquency in the post-crisis period. Because the specification
contains no lagged dependent variable, it cannot correct a persistent level error, and this
bias carries into the evaluation window. Residuals are close to normal (Jarque-Bera
p=0.19) but strongly autocorrelated, so Newey-West standard errors are used for inference
throughout.

In [ ]:
# 4.1 - OLS fitter
def fit_ols(mats):
    """OLS with Newey-West HAC standard errors, maxlags=4 (one year at quarterly
    frequency). Fits twice: on the training block for evaluation, and on the full history
    for the production forecast. The evaluation fit never sees 2021 onward."""
    X_tr, y_tr, idx_tr = mats['train']
    X_ev, y_ev, idx_ev = mats['eval']
    X_fu, y_fu, _      = mats['full']
    X_fc, idx_fc       = mats['fcst']

    add = lambda X: sm.add_constant(X, has_constant='add')

    model = sm.OLS(y_tr, add(X_tr)).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    prod  = sm.OLS(y_fu, add(X_fu)).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

    return {'model': model, 'prod_model': prod,
            'train_pred': model.predict(add(X_tr)),
            'eval_pred':  model.predict(add(X_ev)),
            'fcst_pred':  prod.predict(add(X_fc)),
            'idx_train': idx_tr, 'idx_eval': idx_ev, 'idx_fcst': idx_fc,
            'y_train': y_tr, 'y_eval': y_ev,
            'best_params': {}, 'val_mae': np.nan}

print('fit_ols defined.')

In [ ]:
# 4.2 - sklearn fitter with inner tune/validation split
def fit_sklearn_model(estimator, param_grid, mats, label='', scale=True):
    """Grid search on the inner split, refit, then forecast.

    Hyperparameters are searched by fitting on 1991-2016 and scoring on 2017-2020, so the
    evaluation window is never used to choose a configuration. The winning configuration
    is refit on the whole training block for evaluation, and separately on the full
    history for the production forecast. Scalers are fitted on training data only.
    """
    X_tr, y_tr, idx_tr = mats['train']
    X_tu, y_tu, _      = mats['tune']
    X_va, y_va, _      = mats['val']
    X_ev, y_ev, idx_ev = mats['eval']
    X_fu, y_fu, _      = mats['full']
    X_fc, idx_fc       = mats['fcst']

    sc = StandardScaler().fit(X_tr) if scale else None
    t  = (lambda X: sc.transform(X)) if scale else (lambda X: X)

    best_mae, best_params = np.inf, {}
    for params in ParameterGrid(param_grid):
        est = clone(estimator).set_params(**params)
        est.fit(t(X_tu), y_tu)
        m = mean_absolute_error(y_va, est.predict(t(X_va)))
        if m < best_mae:
            best_mae, best_params = m, params

    # A winner on a grid edge means the search space was too narrow. Parameters with a hard
    # theoretical bound cannot be widened past it, so selecting that value is not evidence
    # of a truncated grid.
    HARD_BOUNDS = {'l1_ratio': 1.0, 'subsample': 1.0}
    for key, grid in param_grid.items():
        nums = [v for v in grid if isinstance(v, (int, float))]
        if len(nums) != len(grid) or len(grid) < 2:
            continue
        won = best_params.get(key)
        if won in (min(nums), max(nums)) and won != HARD_BOUNDS.get(key):
            print(f'  BOUNDARY: {label} {key}={won} at grid edge')

    final = clone(estimator).set_params(**best_params).fit(t(X_tr), y_tr)

    sc_p = StandardScaler().fit(X_fu) if scale else None
    t_p  = (lambda X: sc_p.transform(X)) if scale else (lambda X: X)
    prod = clone(estimator).set_params(**best_params).fit(t_p(X_fu), y_fu)

    return {'model': final, 'prod_model': prod, 'scaler': sc,
            'train_pred': final.predict(t(X_tr)),
            'eval_pred':  final.predict(t(X_ev)),
            'fcst_pred':  prod.predict(t_p(X_fc)),
            'idx_train': idx_tr, 'idx_eval': idx_ev, 'idx_fcst': idx_fc,
            'y_train': y_tr, 'y_eval': y_ev,
            'best_params': best_params, 'val_mae': round(best_mae, 4)}

print('fit_sklearn_model defined.')

In [ ]:
# 4.3 - loss and skill functions
def mae(y, f):
    """Mean absolute error, in percentage points of the delinquency rate."""
    return mean_absolute_error(y, f)

def rmse(y, f):
    """Root mean squared error. Penalises large errors more heavily than MAE."""
    return np.sqrt(mean_squared_error(y, f))

def smape(y, f):
    """Symmetric mean absolute percentage error. Scale-free, bounded 0-200%."""
    y, f = np.asarray(y), np.asarray(f)
    return 100 * np.mean(np.abs(y - f) / ((np.abs(y) + np.abs(f)) / 2))

def skill(y, f, ref, loss='mae'):
    """Forecast skill against a reference forecast: SS = 1 - L(model)/L(reference).

    Positive means the model beats the reference, zero means equal, negative means worse.
    Read as fractional loss reduction: +0.15 is 15% lower loss than the reference.
    The reference must be constructible at the forecast origin, so no reference here uses
    information from the evaluation window itself.
    """
    y, f, ref = np.asarray(y), np.asarray(f), np.asarray(ref)
    if loss == 'mae':
        num, den = mae(y, f), mae(y, ref)
    elif loss == 'mse':
        num, den = np.mean((y - f) ** 2), np.mean((y - ref) ** 2)
    else:
        raise ValueError("loss must be 'mae' or 'mse'")
    return np.nan if den <= 0 else 1 - num / den

print('Loss and skill functions defined.')

In [ ]:
# 4.4 - hyperparameter grids
# Ridge, Lasso and Elastic Net all select the smallest penalty available and converge on
# the OLS solution. Extending those grids downward is not informative, so they are left
# as-is and the boundary hit is reported as a finding rather than a grid defect.
GRIDS = {
    'Ridge':         (Ridge(),
                      {'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1000.0]}),
    'Lasso':         (Lasso(max_iter=50000),
                      {'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0]}),
    'Elastic Net':   (ElasticNet(max_iter=50000),
                      {'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
                       'l1_ratio': [0.05, 0.2, 0.5, 0.8, 0.95, 1.0]}),
    'KRR':           (KernelRidge(kernel='rbf'),
                      {'alpha': [1e-3, 1e-2, 0.1, 1.0, 10.0],
                       'gamma': [1e-4, 1e-3, 1e-2, 0.1, 1.0]}),
    'SVR':           (SVR(kernel='rbf'),
                      {'C': [0.1, 1.0, 10.0, 100.0, 1000.0],
                       'epsilon': [1e-3, 1e-2, 0.1, 0.5, 1.0, 2.0],
                       'gamma': ['scale']}),
    'Random Forest': (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
                      {'max_depth': [2, 3, 4, 6, 8, None],
                       'min_samples_leaf': [1, 2, 4, 8, 16]}),
    'XGBoost':       (xgb.XGBRegressor(n_estimators=300, random_state=SEED,
                                       verbosity=0, n_jobs=-1),
                      {'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.3, 0.5],
                       'max_depth': [1, 2, 3, 4, 6],
                       'subsample': [0.6, 0.8, 1.0]}),
    # Single hidden layer, 4-8 units, strong L2, early stopping. Grinsztajn et al. (2022)
    # find trees competitive with deep models at ~10,000 samples; we have 117.
    'MLP':           (MLPRegressor(max_iter=5000, early_stopping=True,
                                   n_iter_no_change=50, random_state=SEED),
                      {'hidden_layer_sizes': [(4,), (6,), (8,)],
                       'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]}),
}

MODEL_NAMES = ['OLS'] + list(GRIDS.keys())
print(f'{len(MODEL_NAMES)} models: {MODEL_NAMES}')
for name, (_, g) in GRIDS.items():
    print(f'  {name:<14} {len(list(ParameterGrid(g))):>3} candidates')

In [ ]:
# 4.5 - fit all nine on the primary specification
results = {}

print('Fitting OLS...')
results['OLS'] = fit_ols(mats_primary)
print(f'  R2={results["OLS"]["model"].rsquared:.3f} | '
      f'DW={durbin_watson(results["OLS"]["model"].resid):.3f}')

for name, (est, grid) in GRIDS.items():
    print(f'Fitting {name}...')
    results[name] = fit_sklearn_model(est, grid, mats_primary, label=name)
    print(f'  {results[name]["best_params"]} | val MAE={results[name]["val_mae"]}')

print(f'\nAll {len(results)} models fitted on the primary specification.')

In [ ]:
# 4.6 - Figure 4.1 training fit, four representative models
# Tree ensembles and the network can fit 117 training quarters closely while generalising
# poorly. Comparing training fit against the validation MAE in 4.5 shows which models are
# overfitting, which the evaluation-window figures in Section 5 cannot reveal.
SHOW = ['OLS', 'Random Forest', 'XGBoost', 'MLP']

fig = make_subplots(rows=2, cols=2, subplot_titles=SHOW,
                    vertical_spacing=0.13, horizontal_spacing=0.08)

for i, name in enumerate(SHOW):
    r, c = i // 2 + 1, i % 2 + 1
    res  = results[name]
    ix   = res['idx_train']
    fig.add_trace(go.Scatter(x=ix, y=res['y_train'], mode='lines', name='Actual',
                             line=dict(color=NAVY, width=1.7),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=res['train_pred'], mode='lines', name='Fitted',
                             line=dict(color=TEAL, width=1.4, dash='dot'),
                             showlegend=(i == 0)), row=r, col=c)
    val = res['val_mae']
    fig.add_annotation(
        x=0.03, y=0.94, xref='x domain', yref='y domain', row=r, col=c,
        text=(f'train MAE={mae(res["y_train"], res["train_pred"]):.3f}'
              + ('' if np.isnan(val) else f' | val MAE={val:.3f}')),
        showarrow=False, font=dict(size=9, color=NAVY),
        bgcolor='rgba(255,255,255,0.8)', bordercolor=NAVY, borderwidth=0.7)

fig.update_layout(
    title=dict(text=('<b>Figure 4.1 - Training Fit by Model (1991 Q1 to 2020 Q4)</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'A close training fit paired with a high validation MAE indicates '
                     'overfitting rather than skill</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.97, yanchor='top'),
    template=TEMPLATE, height=620,
    legend=dict(orientation='h', yanchor='top', y=1.07, xanchor='center', x=0.5),
    margin=dict(t=120, b=45))
fig.update_yaxes(title_text='Delinquency Rate (%)', col=1)
fig.show()

In [ ]:
# 4.7 - Figure 4.2 OLS residual diagnostics
ols_m = results['OLS']['model']
resid = ols_m.resid
ix_tr = results['OLS']['idx_train']
dw    = durbin_watson(resid)
lb    = acorr_ljungbox(resid, lags=[4, 8], return_df=True)
jb_p  = stats.jarque_bera(resid)[1]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                    subplot_titles=['Residuals over time',
                                    'Residual distribution vs normal'])

fig.add_trace(go.Scatter(x=ix_tr, y=resid, mode='lines+markers', showlegend=False,
                         line=dict(color=BLUE, width=1.1), marker=dict(size=3)),
              row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color=GREY, line_width=1, row=1, col=1)

fig.add_trace(go.Histogram(x=resid, nbinsx=25, histnorm='probability density',
                           marker_color=BLUE, opacity=0.6, showlegend=False),
              row=1, col=2)
grid = np.linspace(resid.min(), resid.max(), 200)
fig.add_trace(go.Scatter(x=grid, y=stats.norm.pdf(grid, resid.mean(), resid.std()),
                         mode='lines', line=dict(color=RED, width=1.5),
                         showlegend=False), row=1, col=2)

fig.update_layout(
    title=dict(text=('<b>Figure 4.2 - OLS Residual Diagnostics</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     f'DW={dw:.3f} | Ljung-Box p(4)={lb["lb_pvalue"].iloc[0]:.4f} | '
                     f'Jarque-Bera p={jb_p:.4f} | residuals run negative after 2011, '
                     'a level shift the specification cannot absorb</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    template=TEMPLATE, height=400, margin=dict(t=115, b=50))
fig.update_yaxes(title_text='Residual (pp)', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=2)
fig.show()

print(f'DW={dw:.4f} | LB p(4)={lb["lb_pvalue"].iloc[0]:.4f}, '
      f'p(8)={lb["lb_pvalue"].iloc[1]:.4f} | Jarque-Bera p={jb_p:.4f}')

## Section 5 - Evaluation

Models are scored on forecasts from the single 2020 Q4 origin. Two windows are reported:
the full 20 quarters, and the clean 14-quarter sub-window from 2022 Q3, where the target
is genuinely observed rather than spline-imputed. The clean sub-window is the primary
reporting window; full-window figures are shown for completeness.

**Metrics.** MAE is primary, in percentage points of the delinquency rate. RMSE weights
large errors more heavily, and comparing the two indicates whether errors are uniform or
driven by a few quarters. sMAPE is scale-free. Skill scores express accuracy relative to a
reference forecast: `1 - loss(model)/loss(reference)`, so +0.15 means 15% lower loss than
the reference and negative means worse. Both references are constructed from training data
only. Train MAE is included so that a low training error paired with a high evaluation
error identifies overfitting.

**Benchmarks.** Three, all constructible at the forecast origin: naive persistence, which
holds the last observed value; the prevailing training mean; and an unemployment-only
regression, the simplest defensible macro specification. Note the persistence anchor is
2020 Q4, which falls inside the spline window, so the benchmark is itself a reconstructed
value.

**Significance testing.** The Diebold-Mariano test compares each model against persistence.
Its truncation lag uses the automatic bandwidth `4*(T/100)^(2/9)` rather than the forecast
horizon: with a single origin rather than rolling origins, a horizon-based lag is not
well defined, and setting it to the window length collapses the small-sample correction to
zero. Power is low regardless, since one error path over 14 correlated quarters carries far
less information than 14 independent comparisons, so the sign test is reported alongside as
a finite-sample companion. Read the skill scores as the substantive comparison and the
p-values as a check on whether differences are distinguishable from noise.

**One confirmatory test.** OLS against naive persistence on the clean sub-window, at an
uncorrected 5%. Every other comparison is exploratory. With nine models the chance that at
least one beats the benchmark by luck is high, and treating all comparisons as confirmatory
would overstate the evidence (Varma and Simon 2006).

**Table 5.2** reports cumulative MAE at horizons 1, 4, 8, 12 and 20, showing how accuracy
develops with distance from the origin rather than averaging it away.

**Figure 5.1** plots each model's forecast against actual, with persistence for reference.
The amber band marks the six quarters scored against imputed values.

In [ ]:
# 5.1 - Diebold-Mariano and sign test
def dm_test(y, f1, f2, lag=None, alpha=0.05):
    """Diebold-Mariano test under absolute-error loss, Bartlett-weighted HAC variance.
    H0: equal expected loss. A negative statistic means f1 is more accurate than f2.

    The truncation lag follows the automatic bandwidth 4*(T/100)^(2/9), which depends on
    sample size rather than forecast horizon. A horizon-based lag is undefined here: with
    one origin rather than rolling origins, setting it to the window length collapses the
    small-sample correction to zero. Power is low either way, so the sign test is the more
    reliable companion.
    """
    y, f1, f2 = np.asarray(y), np.asarray(f1), np.asarray(f2)
    d = np.abs(y - f1) - np.abs(y - f2)
    T = len(d)

    if np.allclose(d, 0):
        return dict(stat=np.nan, p=np.nan, sig=False, lag=0, note='self-comparison')

    if lag is None:
        lag = int(np.floor(4 * (T / 100) ** (2 / 9)))
    lag = max(0, min(lag, T - 2))

    e   = d - d.mean()
    var = np.dot(e, e) / T
    for k in range(1, lag + 1):
        var += 2 * (1 - k / (lag + 1)) * np.dot(e[k:], e[:-k]) / T
    var /= T

    if var <= 0:
        return dict(stat=np.nan, p=np.nan, sig=False, lag=lag,
                    note='non-positive HAC variance')

    stat = d.mean() / np.sqrt(var)
    p    = 2 * (1 - stats.t.cdf(abs(stat), df=T - 1))
    return dict(stat=round(stat, 4), p=round(p, 4), sig=p < alpha, lag=lag, note='')

def sign_test(y, f1, f2):
    """Counts quarters where f1 has lower absolute error than f2; under H0 the count is
    Binomial(T, 0.5). Returns NaN for a self-comparison, where all differentials are zero
    and the count would be spuriously significant."""
    d = np.abs(y - np.asarray(f1)) - np.abs(y - np.asarray(f2))
    T = len(d)
    if np.allclose(d, 0):
        return 0, T, np.nan
    wins = int((d < 0).sum())
    p = min(1.0, 2 * min(stats.binom.cdf(wins, T, 0.5),
                         1 - stats.binom.cdf(wins - 1, T, 0.5)))
    return wins, T, round(p, 4)

print(f'DM automatic lag: {int(np.floor(4*(14/100)**(2/9)))} at T=14, '
      f'{int(np.floor(4*(20/100)**(2/9)))} at T=20')

In [ ]:
# 5.2 - benchmarks
X_ev, y_ev, idx_ev = mats_primary['eval']
y_tr               = mats_primary['train'][1]
idx_ev             = pd.DatetimeIndex(idx_ev)
clean_mask         = (idx_ev >= CLEAN_START) & (idx_ev <= CLEAN_END)

# All three are constructible at the 2020 Q4 origin.
bm_persist = np.full(len(y_ev), y_tr[-1])      # random walk: last observed value held
bm_mean    = np.full(len(y_ev), y_tr.mean())   # prevailing training mean

# Unemployment-only regression: the simplest defensible macro model, and the usual
# single-variable satellite specification in practice.
u_tr = df_train[['us_unemployment_L0', TARGET, 'covid_dummy']].dropna()
ols_u = sm.OLS(u_tr[TARGET].values,
               sm.add_constant(u_tr[['us_unemployment_L0', 'covid_dummy']].values,
                               has_constant='add')
               ).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
bm_unemp = ols_u.predict(sm.add_constant(
    df_eval.loc[idx_ev, ['us_unemployment_L0', 'covid_dummy']].values, has_constant='add'))

BENCHMARKS = {'Naive persistence': bm_persist,
              'Long-run mean': bm_mean,
              'Unemployment only': bm_unemp}

# The persistence anchor falls inside the spline window, so the benchmark the confirmatory
# test runs against is itself a reconstructed value rather than an observation.
anchor_date = pd.Timestamp(mats_primary['train'][2][-1])
print(f'Persistence anchor: {y_tr[-1]:.3f}% at {q(anchor_date)} '
      f'(inside spline window: '
      f'{pd.Timestamp(COVID_START) <= anchor_date <= pd.Timestamp(COVID_END)})')
print(f'Long-run mean: {y_tr.mean():.3f}%')
print(f'Unemployment-only R2: {ols_u.rsquared:.3f}')
print(f'Clean sub-window: {clean_mask.sum()} of {len(idx_ev)} evaluation quarters')

In [ ]:
# 5.3 - Table 5.1 metrics on both windows
def metric_row(label, window, y, f, ref_persist, ref_mean):
    wins, T, sign_p = sign_test(y, f, ref_persist)
    dm  = dm_test(y, f, ref_persist)
    err = np.asarray(y) - np.asarray(f)
    lb  = acorr_ljungbox(err, lags=[min(4, len(err) // 2)], return_df=True)
    tr  = (round(mae(results[label]['y_train'], results[label]['train_pred']), 4)
           if label in results else np.nan)
    return {'Model': label, 'Window': window,
            'Train MAE': tr,
            'MAE': round(mae(y, f), 4), 'RMSE': round(rmse(y, f), 4),
            'sMAPE': round(smape(y, f), 2),
            'Skill vs persist': round(skill(y, f, ref_persist, 'mae'), 4),
            'Skill vs mean': round(skill(y, f, ref_mean, 'mse'), 4),
            'LB p': round(lb['lb_pvalue'].iloc[0], 4),
            'DM stat': dm['stat'], 'DM p': dm['p'],
            'DM sig': 'Yes' if dm['sig'] else 'No',
            'Sign': f'{wins}/{T}', 'Sign p': sign_p}

WINDOWS = [('Full (20Q)', slice(None)), ('Clean (14Q)', clean_mask)]

rows = []
for name in MODEL_NAMES:
    r = results[name]
    for wlabel, m in WINDOWS:
        rows.append(metric_row(name, wlabel, r['y_eval'][m], r['eval_pred'][m],
                               bm_persist[m], bm_mean[m]))
for bname, bpred in BENCHMARKS.items():
    for wlabel, m in WINDOWS:
        rows.append(metric_row(bname, wlabel, y_ev[m], bpred[m],
                               bm_persist[m], bm_mean[m]))

metrics_df = pd.DataFrame(rows)
clean_tbl  = (metrics_df[metrics_df['Window'] == 'Clean (14Q)']
              .sort_values('MAE').set_index('Model'))

print('Table 5.1 - Clean sub-window (2022 Q3 to 2025 Q4), primary reporting window')
display(clean_tbl[['Train MAE', 'MAE', 'RMSE', 'sMAPE', 'Skill vs persist',
                   'Skill vs mean', 'DM stat', 'DM p', 'DM sig', 'Sign', 'Sign p']])

In [ ]:
# 5.4 - Table 5.2 per-horizon accuracy
# Cumulative error over the first h quarters from the 2020 Q4 origin. Shows whether
# accuracy degrades with horizon, which a single window-level figure conceals.
h_rows = []
for name in MODEL_NAMES + list(BENCHMARKS):
    pred = results[name]['eval_pred'] if name in results else BENCHMARKS[name]
    row = {'Model': name}
    for h in HORIZONS:
        row[f'h={h}'] = round(mae(y_ev[:h], pred[:h]), 4)
    h_rows.append(row)

horizon_df = pd.DataFrame(h_rows).set_index('Model')
print('Table 5.2 - MAE by forecast horizon (cumulative from 2020 Q4 origin, full window)')
display(horizon_df)

In [ ]:
# 5.5 - Figure 5.1 forecasts against actual
ncols = 3
nrows = int(np.ceil(len(MODEL_NAMES) / ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=MODEL_NAMES,
                    vertical_spacing=0.11, horizontal_spacing=0.07)

for i, name in enumerate(MODEL_NAMES):
    r, c = i // ncols + 1, i % ncols + 1
    res = results[name]
    ix  = pd.DatetimeIndex(res['idx_eval'])
    fig.add_vrect(x0=str(ix[0].date()), x1=CLEAN_START, row=r, col=c,
                  fillcolor='rgba(230,126,34,0.10)', line_width=0)
    fig.add_trace(go.Scatter(x=ix, y=res['y_eval'], mode='lines', name='Actual',
                             line=dict(color=NAVY, width=2),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=res['eval_pred'], mode='lines+markers',
                             name='Forecast', line=dict(color=TEAL, width=1.6, dash='dash'),
                             marker=dict(size=3), showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=bm_persist, mode='lines', name='Persistence',
                             line=dict(color=GREY, width=1, dash='dot'),
                             showlegend=(i == 0)), row=r, col=c)

fig.update_layout(
    title=dict(text=('<b>Figure 5.1 - Evaluation Window Forecasts (2021 Q1 to 2025 Q4)</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'Amber = the 6 quarters scored against spline-imputed values, '
                     'excluded from the primary reporting window</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.975, yanchor='top'),
    template=TEMPLATE, height=260 * nrows,
    legend=dict(orientation='h', yanchor='top', y=1.06, xanchor='center', x=0.5),
    margin=dict(t=125, b=40))
fig.show()

In [ ]:
# 5.6 - pre-registered confirmatory test
# One confirmatory comparison at uncorrected 5%: OLS against naive persistence on the
# clean sub-window. Every other comparison in this notebook is exploratory. With nine
# models the chance that at least one beats the benchmark by luck is high, so treating all
# of them as confirmatory would overstate the evidence (Varma and Simon 2006).
y_c   = y_ev[clean_mask]
ols_c = results['OLS']['eval_pred'][clean_mask]
p_c   = bm_persist[clean_mask]

dm_c = dm_test(y_c, ols_c, p_c)
w, T, sp = sign_test(y_c, ols_c, p_c)

print('=' * 62)
print('PRE-REGISTERED CONFIRMATORY TEST')
print('OLS vs naive persistence | clean 14Q sub-window | alpha=0.05')
print('=' * 62)
print(f'OLS MAE         : {mae(y_c, ols_c):.4f}%')
print(f'Persistence MAE : {mae(y_c, p_c):.4f}%')
print(f'Skill vs persist: {skill(y_c, ols_c, p_c, "mae"):+.4f}')
print(f'DM statistic    : {dm_c["stat"]} (Bartlett HAC, lag={dm_c["lag"]})')
print(f'DM p-value      : {dm_c["p"]}  ->  '
      f'{"SIGNIFICANT" if dm_c["sig"] else "not significant"}')
print(f'Sign test       : OLS wins {w}/{T}, p={sp}')
print('-' * 62)
if dm_c['sig'] and mae(y_c, ols_c) < mae(y_c, p_c):
    print('OLS significantly outperforms naive persistence.')
elif dm_c['sig']:
    print('Difference is significant, but persistence is the more accurate forecast.')
else:
    print('No significant difference from naive persistence. OLS remains the')
    print('deliverable on governance grounds; the null result is reported as such.')
print('=' * 62)

In [ ]:
# 5.7 - SHAP setup
import shap
print(f'shap {shap.__version__}')

In [ ]:
# 5.8 - Table 5.3 SHAP decomposition, OLS vs best challenger
# The best ML model outperforms OLS on the clean sub-window. SHAP attributes each model's
# prediction to its inputs, so the two can be compared on the same basis: does the
# challenger use the same variables differently, or different variables?
best_ml = clean_tbl.drop(index=['OLS'] + list(BENCHMARKS), errors='ignore').index[0]

X_tr  = mats_primary['train'][0]
y_tr  = mats_primary['train'][1]
feats = FEATURES_EQ511

# OLS is linear, so its SHAP values are exact and analytic: beta_j * (x_j - mean(x_j)).
# Both terms are on the original scale, giving contributions in percentage points.
beta     = np.asarray(results['OLS']['model'].params[1:])
shap_ols = (X_tr - X_tr.mean(axis=0)) * beta

# The challenger was fitted on scaled inputs, so it is explained on the same scale.
# KernelExplainer is model-agnostic; the background set is reduced by k-means to keep
# runtime manageable. Output is still in percentage points.
sc      = results[best_ml]['scaler']
Xs      = sc.transform(X_tr)
expl    = shap.KernelExplainer(results[best_ml]['model'].predict, shap.kmeans(Xs, 25))
shap_ml = expl.shap_values(Xs, silent=True)

# SHAP is additive: baseline plus contributions must reconstruct the prediction.
recon = np.abs(shap_ml.sum(axis=1) + expl.expected_value
               - results[best_ml]['train_pred']).max()
print(f'Explained: OLS and {best_ml} | max additivity error {recon:.4f}pp')

def norm(v):
    v = np.clip(v, 0, None)
    return v / v.sum()

imp = pd.DataFrame({
    'Variable': feats,
    'OLS':      norm(np.abs(shap_ols).mean(axis=0)).round(4),
    best_ml:    norm(np.abs(shap_ml).mean(axis=0)).round(4),
})
imp['Rank OLS']        = imp['OLS'].rank(ascending=False).astype(int)
imp[f'Rank {best_ml}'] = imp[best_ml].rank(ascending=False).astype(int)
imp['Rank gap']        = (imp['Rank OLS'] - imp[f'Rank {best_ml}']).abs()
imp = imp.sort_values('OLS', ascending=False)

rho = stats.spearmanr(imp['OLS'], imp[best_ml]).statistic
print(f'\nTable 5.3 - Mean |SHAP| by variable, normalised to sum to 1')
print(f'{best_ml} clean MAE {clean_tbl.loc[best_ml, "MAE"]:.4f} vs '
      f'OLS {clean_tbl.loc["OLS", "MAE"]:.4f} | Spearman rank correlation {rho:.3f}')
display(imp.set_index('Variable'))

In [ ]:
# 5.9 - Figure 5.2 SHAP beeswarm and importance comparison
order = list(imp['Variable'])[::-1]
pos   = {f: i for i, f in enumerate(order)}
short = [f.replace('us_', '').replace('_', ' ') for f in order]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13,
                    column_widths=[0.58, 0.42],
                    subplot_titles=[f'{best_ml}: per-quarter contributions',
                                    'Mean |SHAP|, normalised'])

rng = np.random.RandomState(SEED)
for f in order:
    j  = feats.index(f)
    sv = shap_ml[:, j]
    xv = X_tr[:, j]
    colour = (xv - xv.min()) / (np.ptp(xv) if np.ptp(xv) else 1)
    fig.add_trace(go.Scatter(
        x=sv, y=pos[f] + rng.uniform(-0.22, 0.22, len(sv)),
        mode='markers', showlegend=False,
        marker=dict(size=4.5, color=colour, colorscale='RdYlBu_r', opacity=0.75,
                    showscale=(f == order[-1]),
                    colorbar=dict(title=dict(text='feature<br>value', font=dict(size=9)),
                                  x=0.53, len=0.55, thickness=10,
                                  tickvals=[0, 1], ticktext=['low', 'high'],
                                  tickfont=dict(size=8))),
        hovertemplate='%{x:.3f}pp<extra></extra>'), row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color=GREY, line_width=1, row=1, col=1)

fig.add_trace(go.Bar(x=imp['OLS'][::-1], y=short, orientation='h',
                     name='OLS', marker_color=NAVY, opacity=0.85), row=1, col=2)
fig.add_trace(go.Bar(x=imp[best_ml][::-1], y=short, orientation='h',
                     name=best_ml, marker_color=TEAL, opacity=0.85), row=1, col=2)

fig.update_yaxes(tickmode='array', tickvals=list(range(len(order))),
                 ticktext=short, row=1, col=1)
fig.update_xaxes(title_text='SHAP value (pp)', row=1, col=1)
fig.update_xaxes(title_text='Share of total importance', row=1, col=2)
fig.update_layout(
    title=dict(text=('<b>Figure 5.2 - SHAP Attribution: OLS vs Best Challenger</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'Left: one dot per training quarter, coloured by the variable\'s '
                     'value. Mixed colours on one side indicate a nonlinear effect. '
                     f'Right: agreement between models, Spearman {rho:.2f}</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    template=TEMPLATE, height=480, barmode='group',
    legend=dict(orientation='h', yanchor='top', y=1.09, xanchor='right', x=1.0),
    margin=dict(t=120, b=55, l=150))
fig.show()

In [ ]:
# 6.1 - production forecast, OLS refit on full history
# The evaluation fit used 1991-2020. The production model refits on all history to 2025 Q4
# so the forecast uses every available observation, then projects 2026 Q1 to 2030 Q4 from
# the Stage A macro paths.
ols_prod  = results['OLS']['prod_model']
fcst_ols  = pd.Series(results['OLS']['fcst_pred'],
                      index=pd.DatetimeIndex(results['OLS']['idx_fcst']),
                      name='OLS')

n_prod = int(ols_prod.nobs)
print(f'Production OLS: n={n_prod} | R2={ols_prod.rsquared:.3f} | '
      f'DW={durbin_watson(ols_prod.resid):.3f}')
print(f'\nForecast 2026 Q1 to 2030 Q4:')
print(f'  range     {fcst_ols.min():.3f}% to {fcst_ols.max():.3f}%')
print(f'  start/end {fcst_ols.iloc[0]:.3f}% -> {fcst_ols.iloc[-1]:.3f}% '
      f'({fcst_ols.iloc[-1] - fcst_ols.iloc[0]:+.3f}pp)')
print(f'  last observed (2025 Q4): {df.loc[EVAL_END, TARGET]:.3f}%')
print(f'  quarter-to-quarter mean absolute change: {fcst_ols.diff().abs().mean():.3f}pp')
print(f'  same for 2015-2019 actuals: '
      f'{df.loc["2015":"2019", TARGET].diff().abs().mean():.3f}pp')

In [ ]:
# 6.2 - Table 6.1 Equation 5.11 coefficients
eq511 = pd.DataFrame({
    'Coefficient': ols_prod.params.round(4),
    'Std error (NW)': ols_prod.bse.round(4),
    't': ols_prod.tvalues.round(3),
    'p': ols_prod.pvalues.round(4),
}, index=['const'] + FEATURES_EQ511)

eq511['Prior'] = ['n/a'] + [SIGN_PRIORS.get(f, ('n/a',))[0] for f in FEATURES_EQ511]
eq511['Matches'] = [
    'n/a' if pr in ('n/a', '?') else
    ('Yes' if (b > 0) == (pr == '+') else 'NO')
    for b, pr in zip(eq511['Coefficient'], eq511['Prior'])]

print(f'Table 6.1 - Equation 5.11, production OLS (Newey-West HAC, maxlags=4)')
print(f'Sample 1991 Q1 to 2025 Q4 | n={n_prod} | R2={ols_prod.rsquared:.3f} | '
      f'adj R2={ols_prod.rsquared_adj:.3f}')
display(eq511)

# Stage A validation status carries into the forecast: a regressor whose own Stage A
# forecast failed the DM test supplies uncertain inputs from 2026 onward.
flagged = [f for f in FEATURES_EQ511
           if any(f.startswith(u) for u in UNVALIDATED_STAGE_A)]
print(f'Regressors without Stage A DM validation: {flagged}')

In [ ]:
# 6.3 - Table 6.2 when each regressor switches to Stage A projections
# A lagged regressor draws on observed data for its first few forecast quarters. This
# records the quarter at which each one begins using Stage A projections instead, which is
# where forecast uncertainty rises.
rows = []
for f in FEATURES_EQ511:
    if f == 'covid_dummy':
        continue
    lag = int(f.rsplit('_L', 1)[1])
    # Forecast quarter t uses the base series at t-lag; observed data ends 2025 Q4.
    switch = pd.Timestamp(EVAL_END) + pd.DateOffset(months=3 * (lag + 1))
    switch = switch + pd.offsets.QuarterEnd(0)
    n_obs  = sum(fcst_ols.index < switch)
    rows.append({'Regressor': f, 'Lag': lag,
                 'Observed inputs until': q(switch - pd.offsets.QuarterEnd(1))
                                          if n_obs else 'none',
                 'Forecast quarters on observed data': n_obs,
                 'On Stage A projections': len(fcst_ols) - n_obs})

switch_df = pd.DataFrame(rows).sort_values('Lag', ascending=False)
print('Table 6.2 - Transition from observed to projected regressor inputs')
display(switch_df.set_index('Regressor'))

In [ ]:
# 6.4 - robustness runs
# OLS only, so that differences are attributable to the specification change rather than
# to model class. R2 is additionally compared against the primary specification fitted on
# R2's own 107-quarter sample, isolating specification from sample composition.
rob = {'Primary': fit_ols(mats_primary),
       'R1 (no dummy)': fit_ols(mats_r1),
       'R2 (all 13)': fit_ols(mats_r2),
       'R2 sample control': fit_ols(mats_primary_r2sample),
       'R3 (raw target)': fit_ols(mats_r3)}

rob_rows = []
for name, r in rob.items():
    ix = pd.DatetimeIndex(r['idx_eval'])
    m  = (ix >= CLEAN_START) & (ix <= CLEAN_END)
    fc = pd.Series(r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
    rob_rows.append({
        'Run': name,
        'Train n': len(r['y_train']),
        'MAE (clean)': round(mae(r['y_eval'][m], r['eval_pred'][m]), 4),
        'Skill vs persist': round(skill(r['y_eval'][m], r['eval_pred'][m],
                                        bm_persist[m], 'mae'), 4),
        '2026 Q1': round(fc.iloc[0], 3),
        '2030 Q4': round(fc.iloc[-1], 3),
        'Max gap vs primary': round(
            np.abs(fc.values - rob['Primary']['fcst_pred']).max(), 3),
    })

rob_df = pd.DataFrame(rob_rows).set_index('Run')
print('Table 6.3 - Robustness runs (OLS only)')
display(rob_df)

# A run diverges materially if its forecast departs from the primary path by more than the
# primary model's own clean-window MAE; anything smaller sits inside its error tolerance.
tol = rob_df.loc['Primary', 'MAE (clean)']
lvl = rob['Primary']['fcst_pred'].mean()
print(f'Pre-registered threshold = primary clean MAE = {tol:.4f}pp')
print(f'Mean forecast level = {lvl:.3f}%, so the threshold is {tol/lvl:.0%} of it '
      f'and is lenient by construction.')
for name in rob_df.index[1:]:
    gap = rob_df.loc[name, 'Max gap vs primary']
    print(f'  {name:<20} max gap {gap:.3f}pp ({gap/lvl:>5.1%} of level)  '
          f'{"MATERIAL" if gap > tol else "within tolerance"}')

In [ ]:
# 6.5 - Figure 6.1 production forecast and robustness paths
hist = df[TARGET].loc[:EVAL_END].dropna()
link = lambda s: ([hist.index[-1]] + list(s.index), [hist.iloc[-1]] + list(s.values))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=['Full history and forecast',
                                    'Forecast horizon, all runs'])

x, y = link(fcst_ols)
fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode='lines', name='Actual',
                         line=dict(color=NAVY, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='OLS forecast',
                         line=dict(color=TEAL, width=2.2)), row=1, col=1)
fig.add_hline(y=hist.mean(), line_dash='dot', line_color=GREY, line_width=1, row=1, col=1)

recent = hist.loc['2019':]
fig.add_trace(go.Scatter(x=recent.index, y=recent.values, mode='lines', showlegend=False,
                         line=dict(color=NAVY, width=2)), row=1, col=2)
for (name, r), col in zip(rob.items(), [TEAL, AMBER, BLUE, GREEN, RED]):
    s = pd.Series(r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
    xr, yr = link(s)
    fig.add_trace(go.Scatter(x=xr, y=yr, mode='lines', name=name,
                             line=dict(color=col, width=2 if name == 'Primary' else 1.4,
                                       dash='solid' if name == 'Primary' else 'dash')),
                  row=1, col=2)

fig.update_layout(
    title=dict(text=('<b>Figure 6.1 - Production Forecast to 2030 Q4</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'Left: full history with the primary OLS forecast. '
                     'Right: all robustness runs over the forecast horizon</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    template=TEMPLATE, height=450,
    legend=dict(orientation='h', yanchor='top', y=1.13, xanchor='center', x=0.5),
    margin=dict(t=135, b=50))
fig.update_yaxes(title_text='Delinquency Rate (%)', row=1, col=1)
fig.show()

In [ ]:
# 6.6 - export
OUT_DIR.mkdir(parents=True, exist_ok=True)

fcst_out = pd.DataFrame({'ML_OLS_primary': fcst_ols})
for name, r in rob.items():
    if name != 'Primary':
        fcst_out[f'ML_OLS_{name.split()[0]}'] = pd.Series(
            r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
for name in MODEL_NAMES:
    if name != 'OLS':
        fcst_out[f'ML_{name.replace(" ", "_")}'] = pd.Series(
            results[name]['fcst_pred'], index=pd.DatetimeIndex(results[name]['idx_fcst']))

fcst_path = OUT_DIR / 'ML_PD_forecasts_US_Q.csv'
mets_path = OUT_DIR / 'ML_metrics_US_Q.csv'
fcst_out.round(4).to_csv(fcst_path)
metrics_df.to_csv(mets_path, index=False)

print(f'Written to {OUT_DIR}:')
print(f'  ML_PD_forecasts_US_Q.csv  {fcst_out.shape[0]} rows x {fcst_out.shape[1]} cols')
print(f'  ML_metrics_US_Q.csv       {metrics_df.shape[0]} rows')
print(f'\nFor 03c: the time-series track writes TS_PD_forecasts_US_Q.csv with the same '
      f'index (2026 Q1 to 2030 Q4) and a ML_/TS_ column prefix.')

In [ ]:
# 6.7 - SHAP setup
%pip install shap --quiet
import shap
import warnings
warnings.filterwarnings('ignore')
print(f'shap {shap.__version__}')